<a href="https://colab.research.google.com/github/albijanashala/ML1/blob/main/work/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — Ranking content decline risk

**Deployed paper:** https://albijanashala.github.io/ML1/

**Repository:** https://github.com/albijanashala/ML1

This notebook mirrors the deployed paper section by section. Every number quoted in the paper
is produced by one of the weekly notebooks in `work/notebooks/`; section 7 regenerates them in
one place so any claim on the page can be traced back to a run.

Lane: **Refresh / Content Opportunity Scoring**.

## 1. Question

*The research question and the decision it supports.*

**Which pages should a content reviewer open first?**

FlyRank manages content portfolios across dozens of client brands. Its own March 2026 portfolio
study shows content reaching peak measured health between 61 and 90 days, holding through a
plateau, then falling sharply in the 271-365 day band — and pages refreshed before that drop
performing close to new content while untouched ones do not recover on their own.

That makes the problem a scheduling one. The lever is timing and the constraint is attention:
a reviewer has finite hours and the portfolio has far more pages than hours.

**The decision this supports.** Given one month of measured search performance for every page,
order those pages so the ones most likely to lose impressions next month — weighted by how much
traffic is at stake — appear first.

**What it does not do.** It does not decide what to write, does not act on any page, and does not
claim that refreshing a page recovers its traffic. It is decision-support: an ordered list with a
reason attached to each row.

## 2. Data

*Which release, which tables, date windows, what was excluded and why — public-safe.*

**Release.** The pseudonymised FlyRank internship warehouse, roughly 79 million rows of daily
search performance as Parquet on Hugging Face. Client and content identifiers are hashes; no
names, domains, URLs, titles or queries exist in the release.

**Tables.**

| Table | One row is | Used for |
|---|---|---|
| `fact_content_daily_performance` | one page on one date | all performance features and the label |
| `dim_content` | one page | word count, age, keyword context, intent |
| `dim_clients` | one client | history coverage, checked for the limitations |

**Windows.** Features from **February 2026**, label over **March 2026**, separated by the calendar
rather than by a cut inside one month. Decision moment: 28 February 2026. The most recent month in
the release is left untouched — it is the natural outcome window for any past-to-future label, so
developing label logic there would mean tuning against an answer key.

**Eligibility.** At least 50 February impressions, which leaves **93,654 pages across 38 clients**.
February holds 54 clients in raw data, so the gate removes 16 entirely; coverage among the
remaining 38 is uneven (median 532 pages per client, minimum 2, maximum 20,358).

**Excluded, and why.**

- Anything measured in March other than the label — a page's outcome cannot predict itself.
- `content_updated_date` and `last_optimized_date` — `dim_content` is a current-state snapshot, so
  a page updated in March carries a March date. These encode the label window while looking like
  ordinary metadata.
- Position columns — inspected day by day they return values below 1 (3,519 impressions against a
  position sum of 524 gives 0.149). Position 1 is the best rank Google awards, so the column failed
  inspection and was dropped.
- GA4 engagement columns — only 4.2% of March rows carry availability, so values would be missing
  for roughly 96% of pages. A binary availability flag is kept; the values are not.
- Identifiers — used for grouping and joining only.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label.** Declining when the March daily impression rate falls below 80% of the February daily
rate. February has 28 days and March has 31, so the comparison is on daily rates; comparing raw
totals would make almost every page look like it grew. Base rate **27.6%** (25,887 pages).

This is a proxy for "needs a reviewer's attention", not a record of whether anyone should have
acted. No editorial outcome exists in the data.

**Features.** 23 model-ready columns in four groups, all measurable on 28 February: volume
(impressions, clicks, active days), rates (CTR, impressions per active day), momentum (second-half
February divided by first half), and content context (word count, age, search volume, competition,
backlinks, plus encoded content type and intent).

Missing values are filled with a stated rule rather than dropped, each paired with a flag where the
absence itself carries information. No rows are lost.

**Baseline.** A hand-written rule on the same pages and the same label: 60% momentum loss, 25%
inconsistent visibility, 15% content age. Readable in one line, needs no training.

**Validation.** Five-fold cross-validation grouped by `client_hash_id`. Every page of a client sits
entirely in train or entirely in test. Pages from one client share a site, a topic mix and a traffic
pattern, so a random row split would let the model recognise portfolios rather than learn decline.

**Leakage checks.** Correlation with the label peaks at 0.124. No feature scores above 0.90 alone.
Every feature's source window is asserted in code. The control: March impressions alone — the
label's own ingredient — score **0.767**, *below* the 0.90 screening threshold, so a leaking column
would have passed the single-feature check undetected. The window audit is what catches it, because
it tests provenance rather than predictive strength.

**Why the model stops at a random forest.** Gradient boosting was not tested. With 23 features and a
proxy label, the plausible gain sits inside the fold-to-fold spread already measured (+/-0.142 on
Precision@50), so a higher mean would not be distinguishable from which clients landed in the test
fold. The logistic regression losing to the hand rule points the same way: the constraint is signal,
not model capacity.

## 4. Results (vs baseline)

*Model vs baseline on the same split.*

**Five-fold GroupKFold on `client_hash_id`. Same pages, same label, same folds. Base rate 0.276.**

| Method | ROC AUC | Average precision | Precision@50 |
|---|---|---|---|
| Hand-written rule | 0.633 +/- 0.085 | 0.444 +/- 0.189 | 0.680 +/- 0.228 |
| Logistic regression | 0.592 +/- 0.055 | 0.373 +/- 0.132 | 0.648 +/- 0.185 |
| **Random forest** | **0.685 +/- 0.045** | **0.486 +/- 0.161** | **0.836 +/- 0.142** |

**The split design is worth more than the model choice.** The same forest under a naive random row
split:

| Split design | ROC AUC | Avg precision | Precision@50 |
|---|---|---|---|
| Random rows | 0.794 +/- 0.003 | 0.668 +/- 0.005 | 1.000 +/- 0.000 |
| Grouped by client | 0.685 +/- 0.045 | 0.486 +/- 0.161 | 0.836 +/- 0.142 |
| Overstatement | +0.109 | +0.182 | +0.164 |

A perfect Precision@50 with zero variance across five folds is a warning, not an achievement: the
five folds were not five experiments, because each contained the same clients.

**A simple model loses to the rule.** Logistic regression at 0.592 against the rule's 0.633 says the
relationship is not linear, and that three hand-chosen signals carry more together than a linear
combination of 23 features does.

**What the model leans on.** Permutation importance puts February momentum first at 0.0525, roughly
twice impressions per active day (0.0233), with word count third (0.0154).

**Where it is wrong.** On 20,358 held-out pages: 300 true positives, 281 false positives, 4,651
false negatives — a recall of about 6%. The misses share a profile: median February momentum of
**1.182**, meaning they were growing through the feature window and then turned. True positives were
already falling at 0.405. The model finds pages that were already sliding; it cannot find pages that
turn.

## 5. Limitations

*What this work cannot claim.*

Every claim is an association measured in one portfolio over two months. Nothing here demonstrates
causation, and nothing describes how Google ranks pages.

- **It ranks; it does not cover.** Recall of roughly 6% makes this a priority order, not an
  inventory of at-risk content.
- **It cannot see pages that turn.** A single-month feature window only detects decline that had
  already started.
- **Sixteen clients are absent entirely.** The eligibility gate removes them, so the queue is silent
  about those portfolios rather than saying their pages are healthy.
- **Coverage is uneven.** One client accounts for roughly 22% of the queue.
- **One month predicts the next.** Seasonality, algorithm updates and one-off events inside the
  window are invisible.
- **The label is a proxy.** An impression drop stands in for "needs attention".
- **No ranking diagnosis.** The position field failed inspection and was excluded.
- **Model and rule are close.** The margin is directional; the rule remains a reasonable fallback.
- **Refresh impact is not established.** That would need an experiment with a control group.

**What would change the result.** Not a stronger model but a longer feature window. Three months of
prior momentum rather than two halves of one would give the model the history it lacks — and the
misses say why: their median February momentum was 1.182, growing right up to the moment the window
closed. The warehouse supports it; per-client history runs from January 2025 to June 2026.

## 6. Ranked recommendations

*The action playbook output.*

The queue orders pages by decline probability weighted by `log1p(February impressions)`. Ranking on
probability alone put pages with 55 to 600 impressions and no clicks at the top — the model was
right they would decline, but nobody gains from fixing them. Raw impressions would collapse the
queue into a volume ranking. The log keeps risk as the driver and lets the size of the loss break
ties. The top fifty carry roughly **699,000** February impressions.

| Reason code | Action | Pages |
|---|---|---|
| Model signal only | manual triage | 29,332 |
| Weak click capture | snippet review | 23,348 |
| Momentum loss | investigate decline | 19,946 |
| Intermittent visibility | check indexing | 15,485 |
| Ageing content | content refresh | 5,543 |

1. **Start at the top and stop when the week ends.** Reliable at the head, thin in the tail.
2. **Read the columns, not the label.** Codes are evaluated in order and momentum is checked first,
   so every page in the top twenty carries the same code even where age or click capture is also
   poor.
3. **Confirm the decline is real before calling it decay.** A steep one-month drop is as consistent
   with deindexing, a redirect or cannibalisation as with staleness.
4. **Treat the flagged pages differently.** 22,004 pages (23.5%) sit more than 40 percentile points
   apart between model and rule. None are in the top twenty.
5. **Do not automate anything downstream.** At 6% recall the evidence does not support irreversible
   actions.
6. **Rebuild monthly and watch four numbers.** Base rate, momentum distribution, disagreement share,
   clients passing the gate.

## 7. Artifacts the paper embeds

*Generate the numbers the paper quotes.*

The paper's figures are inline SVG built by hand, so there are no chart files to regenerate. What
the paper does depend on is its numbers. The cells below rebuild the frame from the warehouse and
write every quoted figure to `work/outputs/capstone_paper_numbers.json`, so any claim on the
deployed page can be checked against a run.

In [1]:
%pip -q install duckdb

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected")

Connected


In [2]:
DECISION_DATE = "2026-02-28"

features = con.sql(f"""
    WITH feb AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions) AS imp_feb,
               SUM(gsc_clicks) AS clicks_feb,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_visible_feb,
               SUM(CASE WHEN report_date <= DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h1,
               SUM(CASE WHEN report_date >  DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h2,
               MAX(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS has_ga4
        FROM {FEB}
        GROUP BY 1, 2
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_mar
        FROM {MAR}
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.imp_feb,
        f.clicks_feb,
        f.days_visible_feb,
        ROUND(f.clicks_feb::FLOAT / NULLIF(f.imp_feb, 0), 5) AS ctr_feb,
        ROUND(f.imp_feb::FLOAT / NULLIF(f.days_visible_feb, 0), 2) AS imp_per_active_day,
        ROUND(f.imp_h2::FLOAT / NULLIF(f.imp_h1, 0), 3) AS trend_within_feb,
        f.has_ga4,
        COALESCE(d.word_count, 0) AS word_count,
        CASE WHEN d.word_count IS NULL THEN 1 ELSE 0 END AS word_count_missing,
        COALESCE(d.search_volume, 0) AS search_volume,
        COALESCE(d.competition, 0) AS competition,
        COALESCE(d.backlinks, 0) AS backlinks,
        DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') AS content_age_days,
        COALESCE(d.content_type, 'unknown') AS content_type,
        COALESCE(d.competition_level, 'unknown') AS competition_level,
        COALESCE(d.main_intent, 'unknown') AS main_intent,
        COALESCE(m.imp_mar, 0) AS imp_mar
    FROM feb f
    LEFT JOIN mar m USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE f.imp_feb >= 50
    ORDER BY f.client_hash_id, f.content_hash_id
""").df()

clients_raw_feb = con.sql(f"SELECT COUNT(DISTINCT client_hash_id) FROM {FEB}").fetchone()[0]
print(f"Rows: {len(features):,}")
print(f"Clients in frame: {features['client_hash_id'].nunique()}")
print(f"Clients in raw February: {clients_raw_feb}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 93,654
Clients in frame: 38
Clients in raw February: 54


In [3]:
df = features.copy()

# February has 28 days and March has 31, so compare daily rates rather than raw totals
FEB_DAYS, MAR_DAYS = 28, 31
df["is_declining"] = ((df["imp_mar"] / MAR_DAYS) < 0.8 * (df["imp_feb"] / FEB_DAYS)).astype(int)

# Stated fills, each paired with a flag where the absence itself carries information
df["no_h1_impressions"] = df["trend_within_feb"].isna().astype(int)
df["trend_within_feb"] = df["trend_within_feb"].fillna(1.0)
df["content_age_days"] = df["content_age_days"].fillna(-1)

competition_order = {"unknown": 0, "LOW": 1, "MEDIUM": 2, "HIGH": 3}
df["competition_level_ord"] = df["competition_level"].map(competition_order).fillna(0).astype(int)
one_hot = pd.get_dummies(df[["content_type", "main_intent"]], prefix=["ctype", "intent"], dtype=int)
df = pd.concat([df, one_hot], axis=1)

drop_cols = ["client_hash_id", "content_hash_id", "imp_mar", "is_declining",
             "content_type", "competition_level", "main_intent"]
model_features = [c for c in df.columns if c not in drop_cols]

print(f"Rows: {len(df):,}")
print(f"Positive rate: {df['is_declining'].mean():.1%} ({df['is_declining'].sum():,} declining)")
print(f"Model-ready features: {len(model_features)}")

Rows: 93,654
Positive rate: 27.6% (25,887 declining)
Model-ready features: 23


In [4]:
import json
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

OUT_DIR = Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

X, y = df[model_features], df["is_declining"]
groups = df["client_hash_id"]


def precision_at_k(y_true, scores, k=50):
    top = np.argsort(scores)[::-1][:k]
    return float(np.asarray(y_true)[top].mean())


# Out-of-fold scores: every page ranked by a model that never saw its own client
df["model_score"] = np.nan
fold_auc, fold_ap, fold_p50 = [], [], []

for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
    model = RandomForestClassifier(n_estimators=200, min_samples_leaf=20,
                                   random_state=42, n_jobs=-1)
    model.fit(X.iloc[tr], y.iloc[tr])
    p = model.predict_proba(X.iloc[te])[:, 1]
    df.iloc[te, df.columns.get_loc("model_score")] = p
    fold_auc.append(roc_auc_score(y.iloc[te], p))
    fold_ap.append(average_precision_score(y.iloc[te], p))
    fold_p50.append(precision_at_k(y.iloc[te], p))

df["priority_score"] = (df["model_score"] * np.log1p(df["imp_feb"])).round(4)
df["impressions_at_risk"] = (df["model_score"] * df["imp_feb"]).round(0)
pages_per_client = groups.value_counts()

paper_numbers = {
    "deployed_paper": "https://albijanashala.github.io/ML1/",
    "feature_window": "2026-02",
    "label_window": "2026-03",
    "pages_scored": int(len(df)),
    "clients_in_frame": int(groups.nunique()),
    "clients_in_raw_february": int(clients_raw_feb),
    "model_ready_features": len(model_features),
    "base_rate": round(float(y.mean()), 4),
    "declining_pages": int(y.sum()),
    "random_forest_grouped": {
        "roc_auc": [round(float(np.mean(fold_auc)), 3), round(float(np.std(fold_auc)), 3)],
        "average_precision": [round(float(np.mean(fold_ap)), 3), round(float(np.std(fold_ap)), 3)],
        "precision_at_50": [round(float(np.mean(fold_p50)), 3), round(float(np.std(fold_p50)), 3)],
    },
    "impressions_at_risk_top50": int(df.nlargest(50, "priority_score")["impressions_at_risk"].sum()),
    "pages_per_client": {
        "median": int(pages_per_client.median()),
        "min": int(pages_per_client.min()),
        "max": int(pages_per_client.max()),
    },
}

with open(OUT_DIR / "capstone_paper_numbers.json", "w") as f:
    json.dump(paper_numbers, f, indent=2)

print(json.dumps(paper_numbers, indent=2))

{
  "deployed_paper": "https://albijanashala.github.io/ML1/",
  "feature_window": "2026-02",
  "label_window": "2026-03",
  "pages_scored": 93654,
  "clients_in_frame": 38,
  "clients_in_raw_february": 54,
  "model_ready_features": 23,
  "base_rate": 0.2764,
  "declining_pages": 25887,
  "random_forest_grouped": {
    "roc_auc": [
      0.685,
      0.045
    ],
    "average_precision": [
      0.486,
      0.161
    ],
    "precision_at_50": [
      0.836,
      0.142
    ]
  },
  "impressions_at_risk_top50": 699020,
  "pages_per_client": {
    "median": 532,
    "min": 2,
    "max": 20358
  }
}


## 8. Five-minute demo outline

*Prepared for the showcase. Timings are targets, not a script.*

**0:00-0:45 — The question**

A content team can review fifty pages a week. The portfolio holds 93,654. Choosing the wrong fifty
costs a month of decay on the right ones. FlyRank's own portfolio study shows content peaking in
measured health between 61 and 90 days and falling sharply past 270, so the lever is timing and the
constraint is attention.

**0:45-2:00 — The method**

Features from February 2026, label from March 2026. The calendar separates them, so no input can see
its own outcome. 23 features on 93,654 pages across 38 clients. A random forest against a
hand-written rule, both scored on the same client-grouped folds — whole clients held out, because
pages from one portfolio share a site, a topic mix and a traffic pattern.

**2:00-3:00 — One chart**

The split demonstrator from section 4 of the paper. Switch it from grouped to random and Precision@50
jumps from 0.836 to 1.000 with zero variance across five folds. That is the chart to show, because
the audience watches the overstatement appear rather than being told about it. The point: a perfect
score with no spread means five folds that were really one experiment.

**3:00-4:00 — One honest result**

Precision@50 of 0.836 sounds strong. Recall is about 6%: 300 true positives against 4,651 false
negatives on held-out clients. Both numbers are true and they describe different things. The misses
share a profile — median February momentum of 1.182, meaning they were growing right up to the
moment the window closed. The model finds pages already sliding; it cannot find pages that turn.

**4:00-5:00 — One recommendation**

Work the top fifty and stop. The ranking is reliable at the head and thin in the tail, and at 6%
recall nothing downstream should be automated — no rewrites, no merges, no deindexing without a
person reading the page. The single highest-value next step is not a stronger model but a longer
feature window.

**Expect this question:** why not gradient boosting? Because the plausible gain sits inside the
+/-0.142 spread already measured, so a higher mean would not be distinguishable from which clients
landed in the test fold.

## 9. Shareable cuts

### Social post — on methodology

> I trained a model to predict which web pages would lose search traffic next month. It scored a
> perfect 1.000 on my top-50 metric, across all five folds, with zero variance.
>
> That was the bug.
>
> My pages belonged to 38 client portfolios. A random train/test split scattered each client's pages
> across both sides, so the model could recognise which portfolios had a bad month instead of
> learning why pages decline. The near-zero variance was the tell: five folds containing the same
> clients aren't five experiments.
>
> Splitting by client instead — whole portfolios held out — dropped it to 0.836, and the spread
> widened to +/-0.142. That width was real uncertainty the first design had been hiding.
>
> The honest number is lower and less quotable. It's also the only one I'd put my name on.
>
> Built on the FlyRank ML Internship dataset. Full write-up and notebooks linked below.

### Employer-facing summary — three sentences

> I built a decision-support queue that ranks 93,654 web pages by the risk their search traffic will
> decline next month, so a content team with capacity for fifty reviews a week knows which fifty to
> open.
>
> It runs on a pseudonymised warehouse of roughly 79 million rows of daily Google Search Console
> performance, with features drawn from one calendar month and the outcome measured in the next,
> validated on client-grouped folds and audited for leakage before any score was reported.
>
> The model observed a mean Precision@50 of 0.836 against a hand-written baseline's 0.680 — and a
> recall of about 6%, which I report alongside it, because the queue is a priority order rather than
> an inventory of risk.

## Self-check

- [x] Every section mirrors the deployed paper
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Demo outline and both shareable cuts are present
- [x] Committed under `work/notebooks/`, paper URL in `submission/paper_url.txt`